In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# ----------------------------
# 1. Load Data
# ----------------------------
df = pd.read_excel("your_data.xlsx")

# Expected columns:
# Sample, Concentration, Abs_sample, Abs_control

# ----------------------------
# 2. Calculate % Inhibition
# ----------------------------
df["Inhibition"] = ((df["Abs_control"] - df["Abs_sample"]) / df["Abs_control"]) * 100

# ----------------------------
# 3. Define Sigmoid Model
# ----------------------------
def sigmoid(x, bottom, top, ic50, hill):
    return bottom + (top - bottom) / (1 + (x/ic50)**hill)

# ----------------------------
# 4. IC50 Calculation per Sample
# ----------------------------
results = []

for sample in df["Sample"].unique():
    sub = df[df["Sample"] == sample]
    
    xdata = sub["Concentration"].values
    ydata = sub["Inhibition"].values
    
    try:
        popt, _ = curve_fit(sigmoid, xdata, ydata, 
                           bounds=([0,0,0,0],[100,100,10,5]))
        
        ic50 = popt[2]
        
        results.append({"Sample": sample, "IC50": ic50})
        
        # Plot
        xfit = np.linspace(min(xdata), max(xdata), 100)
        yfit = sigmoid(xfit, *popt)
        
        plt.figure()
        plt.scatter(xdata, ydata, label="Data")
        plt.plot(xfit, yfit, label=f"IC50={ic50:.3f}")
        plt.xscale("log")
        plt.xlabel("Concentration (mg/mL)")
        plt.ylabel("% Inhibition")
        plt.title(sample)
        plt.legend()
        plt.savefig(f"{sample}_IC50.png")
        plt.close()
        
    except:
        print(f"Fit failed for {sample}")

ic50_df = pd.DataFrame(results)
ic50_df.to_excel("IC50_results.xlsx", index=False)

# ----------------------------
# 5. ANOVA (Compare Samples)
# ----------------------------
model = ols('Inhibition ~ C(Sample)', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

print("\nANOVA RESULTS:")
print(anova_table)

# ----------------------------
# 6. Boxplot (Publication)
# ----------------------------
plt.figure(figsize=(10,6))
sns.boxplot(data=df, x="Sample", y="Inhibition")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("Boxplot.png")

# ----------------------------
# 7. Standard Curve (TPC/FRAP)
# ----------------------------
std = pd.read_excel("standard_curve.xlsx")

slope, intercept, r, p, stderr = stats.linregress(std["Conc"], std["Abs"])

plt.figure()
plt.scatter(std["Conc"], std["Abs"])
plt.plot(std["Conc"], slope*std["Conc"] + intercept)
plt.xlabel("Concentration")
plt.ylabel("Absorbance")
plt.title(f"R² = {r**2:.3f}")
plt.savefig("Standard_curve.png")

print("\nStandard Curve Equation:")
print(f"y = {slope:.4f}x + {intercept:.4f}")